# 🚦 EDA y calidad de datos — US Accidents (2016–2023)

**Curso:** Big Data — Universidad Nacional de Juliaca (UNAJ)  
**Semestre:** 2026-2  
**Proyecto:** Accidentes de tránsito e impacto sobre el tráfico  
**Integrantes:** *Completar antes de entregar*  
**Dataset:** *US Accidents (2016–2023)* — Sobhan Moosavi, Kaggle

> **Alcance de esta entrega:** comprender el activo, describir patrones relevantes y diagnosticar su calidad. No se construyen modelos predictivos ni se hacen afirmaciones causales.


## 1. Pregunta motivadora y utilidad esperada

> ### ¿Cuándo, dónde y bajo qué condiciones un accidente deja de ser un incidente aislado y se convierte en un colapso del tráfico?

Cada accidente puede bloquear vías, retrasar servicios de emergencia y generar congestión para personas no involucradas. Los recursos de control vial, sin embargo, suelen distribuirse de manera uniforme aunque la concentración de incidentes de alto impacto cambia por hora, lugar y condiciones del entorno.

| Pregunta de contexto | Respuesta del proyecto |
|---|---|
| **¿Qué queremos describir?** | Patrones de hora, día, clima, ubicación y elementos viales en los accidentes de severidad 3–4. |
| **¿Qué decisión puede mejorar?** | La asignación de patrullaje, gestión semafórica y servicios de emergencia por franja horaria y zona. |
| **¿Quién se beneficia?** | Departamentos de transporte, policía de tránsito y servicios de emergencia. |
| **Tipo de propósito** | Mejorar una decisión operativa. |

**Criterio operativo:** en este notebook, `Severity` 3–4 se usa como **proxy de alto impacto sobre el tráfico**. No significa necesariamente lesiones graves ni demuestra por sí sola un “colapso” observable.


## 2. Fuente, periodo y condiciones de uso

| Aspecto | Detalle |
|---|---|
| **Fuente** | *US Accidents (2016–2023)*, Sobhan Moosavi — [Kaggle](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents/data) |
| **Origen** | APIs de tráfico que integran reportes de departamentos de transporte, policía, cámaras y sensores viales |
| **Periodo informado** | Febrero de 2016 a marzo de 2023 |
| **Cobertura informada** | 49 estados de Estados Unidos |
| **Tamaño original** | Aproximadamente 7.7 millones de registros, 46 columnas y 3 GB |
| **Archivo esperado** | `data/bronze/US_Accidents_March23.csv` |
| **Condiciones de uso** | CC BY-NC-SA 4.0: uso no comercial, con atribución y obras derivadas bajo la misma licencia |

**Atribución académica sugerida:** Moosavi et al. (2019), *A Countrywide Traffic Accident Dataset*.

### Preparación del archivo

1. Descargar el CSV desde Kaggle.
2. Conservar el nombre `US_Accidents_March23.csv`.
3. Ubicarlo en `data/bronze/` sin modificar el archivo original.

La carga siguiente localiza la raíz del repositorio automáticamente. Para una prueba limitada puede definirse la variable de entorno `US_ACCIDENTS_NROWS`; si no se define, se analiza el archivo completo.


## 3. Unidad de análisis

**Cada fila representa un accidente de tránsito reportado.** La fila registra el inicio y fin del incidente, coordenadas y jurisdicción, condiciones meteorológicas cercanas y elementos viales presentes en el entorno.

La variable `Severity` toma valores de 1 a 4 y representa el **impacto reportado en el tráfico**, no la gravedad clínica de las lesiones. Para la pregunta del proyecto se consideran de alto impacto los registros con severidad 3 o 4.

### Precauciones de granularidad

- `ID` debería identificar un reporte único; se comprobará su unicidad.
- La frecuencia de accidentes no equivale al riesgo de sufrir un accidente: faltan denominadores de exposición como volumen vehicular o kilómetros recorridos.
- El dataset no contiene una clase explícita de vía (autopista, avenida, calle residencial). `Junction`, `Crossing`, `Traffic_Signal`, `Stop`, `Roundabout` y otras banderas se emplean únicamente como **proxies del contexto vial**.


---
## 4. Preparación reproducible

Se utilizan únicamente librerías declaradas en `requirements.txt`. Los gráficos se guardan en `outputs/us_accidents/` y los datos originales permanecen en la capa Bronze.


In [ ]:
# Celda 4.1: importaciones, estilo y rutas portables
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.2f}".format)


def find_project_root(start: Path) -> Path:
    # Localiza la raíz por las carpetas data/ y notebooks/.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto (carpetas data/ y notebooks/)."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "us_accidents"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "navy": "#17324D",
    "blue": "#2E86AB",
    "teal": "#0E7490",
    "gold": "#F4B942",
    "coral": "#E76F51",
    "green": "#2A9D8F",
    "gray": "#64748B",
    "light": "#F4F8FB",
}
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": COLORS["light"],
})


def save_fig(filename: str, fig=None) -> Path:
    # Guarda una figura en el directorio de salidas del proyecto.
    path = OUTPUT_DIR / filename
    target = fig if fig is not None else plt.gcf()
    target.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    print(f"Gráfico guardado: {path.relative_to(PROJECT_ROOT)}")
    return path


print(f"Proyecto : {PROJECT_ROOT}")
print(f"Python   : {sys.version.split()[0]} ({platform.system()})")
print(f"Pandas   : {pd.__version__}")
print(f"Salidas  : {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


### 4.2 Carga del dataset

La lectura selecciona las columnas relacionadas con la pregunta y asigna tipos compactos. El archivo original no se modifica. La variable `US_ACCIDENTS_CSV` permite indicar otra copia del mismo CSV sin editar el notebook.


In [ ]:
# Celda 4.2: configuración de carga reproducible
CSV_NAME = "US_Accidents_March23.csv"
csv_override = os.getenv("US_ACCIDENTS_CSV")
CSV_PATH = Path(csv_override).expanduser() if csv_override else BRONZE_DIR / CSV_NAME

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontró {CSV_PATH}. Descarga el dataset desde Kaggle y ubica "
        f"{CSV_NAME} en {BRONZE_DIR.relative_to(PROJECT_ROOT)}/, o define "
        "la variable de entorno US_ACCIDENTS_CSV con la ruta del archivo."
    )

USECOLS = [
    "ID", "Source", "Severity", "Start_Time", "End_Time",
    "Start_Lat", "Start_Lng", "Distance(mi)",
    "City", "County", "State", "Timezone", "Weather_Timestamp",
    "Temperature(F)", "Humidity(%)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)", "Weather_Condition",
    "Bump", "Crossing", "Junction", "Railway", "Roundabout",
    "Stop", "Traffic_Calming", "Traffic_Signal", "Sunrise_Sunset",
]

DTYPES = {
    "ID": "string",
    "Source": "category",
    "Severity": "Int8",
    "Start_Time": "string",
    "End_Time": "string",
    "Start_Lat": "Float32",
    "Start_Lng": "Float32",
    "Distance(mi)": "Float32",
    "City": "category",
    "County": "category",
    "State": "category",
    "Timezone": "category",
    "Weather_Timestamp": "string",
    "Temperature(F)": "Float32",
    "Humidity(%)": "Float32",
    "Visibility(mi)": "Float32",
    "Wind_Speed(mph)": "Float32",
    "Precipitation(in)": "Float32",
    "Weather_Condition": "category",
    "Bump": "boolean",
    "Crossing": "boolean",
    "Junction": "boolean",
    "Railway": "boolean",
    "Roundabout": "boolean",
    "Stop": "boolean",
    "Traffic_Calming": "boolean",
    "Traffic_Signal": "boolean",
    "Sunrise_Sunset": "category",
}

nrows_env = os.getenv("US_ACCIDENTS_NROWS", "").strip()
NROWS = int(nrows_env) if nrows_env else None
load_scope = f"primeras {NROWS:,} filas (prueba)" if NROWS else "archivo completo"

print(f"Fuente     : {CSV_PATH}")
print(f"Tamaño     : {CSV_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"Alcance    : {load_scope}")

started = time.perf_counter()
df = pd.read_csv(
    CSV_PATH,
    usecols=USECOLS,
    dtype=DTYPES,
    nrows=NROWS,
    low_memory=False,
)
elapsed = time.perf_counter() - started

print(f"Carga lista: {len(df):,} filas x {df.shape[1]} columnas en {elapsed:.1f} s")
print(f"Memoria del DataFrame: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MiB")


---
## 5. Evidencia básica y perfil descriptivo

Esta sección presenta las cuatro evidencias requeridas: `head()`, `shape`, `info()` y `describe()`. Los identificadores se excluyen de `describe()` porque promediarlos o buscar su categoría más frecuente no aporta significado analítico.


In [ ]:
# Celda 5.1: primeras filas
df.head()


In [ ]:
# Celda 5.2: dimensiones observadas
print(f"shape = {df.shape}")
print(f"Filas: {df.shape[0]:,} | Columnas cargadas: {df.shape[1]}")


In [ ]:
# Celda 5.3: tipos, completitud y uso de memoria
df.info(memory_usage="deep")


In [ ]:
# Celda 5.4: perfil descriptivo de variables relevantes
DESCRIBE_COLUMNS = [
    "Severity", "Start_Lat", "Start_Lng", "Distance(mi)",
    "Temperature(F)", "Humidity(%)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)", "State",
    "Weather_Condition", "Sunrise_Sunset",
]
resumen_descriptivo = df[DESCRIBE_COLUMNS].describe(
    include="all", percentiles=[0.25, 0.50, 0.75, 0.95]
).T
display(resumen_descriptivo)


### 5.5 Cómo interpretar `describe()`

| Estadístico | Lectura adecuada |
|---|---|
| `count` | Registros no nulos; una diferencia frente al total evidencia incompletitud. |
| `mean` | Promedio; puede desplazarse por valores extremos. |
| `std` | Dispersión respecto al promedio; no se interpreta para códigos o IDs. |
| `min`, `max` | Extremos útiles para detectar valores físicamente imposibles o atípicos. |
| `25%`, `50%`, `75%`, `95%` | Percentiles; la mediana (`50%`) es más robusta que el promedio. |
| `unique`, `top`, `freq` | Diversidad, categoría modal y frecuencia de esa categoría. |

La celda siguiente convierte el perfil en una lectura breve. Es una descripción de los registros cargados, no una explicación causal.


In [ ]:
# Celda 5.5: interpretación automática y verificable
severity_valid = df["Severity"].dropna()
high_impact_pct = severity_valid.isin([3, 4]).mean() * 100 if len(severity_valid) else np.nan
top_state = df["State"].value_counts(dropna=True)
top_weather = df["Weather_Condition"].value_counts(dropna=True)

print("INTERPRETACIÓN DEL PERFIL")
print("=" * 72)
print(f"• Se cargaron {len(df):,} accidentes y {df.shape[1]} campos seleccionados.")
print(f"• Los accidentes de severidad 3–4 representan {high_impact_pct:.2f}% de severidades válidas.")
if not top_state.empty:
    print(f"• El estado con más reportes es {top_state.index[0]} ({top_state.iloc[0]:,} registros).")
if not top_weather.empty:
    print(
        f"• La condición climática modal es {top_weather.index[0]!s} "
        f"({top_weather.iloc[0]:,} registros)."
    )
print("• Los mínimos, máximos y nulos se validarán con reglas de dominio en la sección 10.")
print("• Estos conteos describen accidentes registrados; no miden exposición vehicular ni causalidad.")


---
## 6. Diccionario de datos de las columnas relevantes

El diccionario distingue identificadores, categorías, fechas, medidas y variables derivadas. Documentar el dominio evita interpretar códigos como cantidades o confundir severidad vial con gravedad clínica.


In [ ]:
# Celda 6.1: diccionario de variables principales y derivadas
diccionario_datos = pd.DataFrame([
    ("ID", "Identificador del reporte de accidente", "Identificador", "Texto no vacío y único", "Evita el doble conteo"),
    ("Source", "Fuente/API que reportó el incidente", "Categórica nominal", "Categorías del proveedor", "Permite evaluar heterogeneidad de reporte"),
    ("Severity", "Impacto del accidente sobre el tráfico", "Ordinal discreta", "Enteros 1, 2, 3 o 4", "Define el proxy de alto impacto (3–4)"),
    ("Start_Time", "Fecha y hora de inicio reportada", "Temporal", "Fecha-hora convertible", "Permite analizar hora, día y periodo"),
    ("End_Time", "Fecha y hora de fin reportada", "Temporal", "Igual o posterior al inicio", "Permite aproximar duración"),
    ("Start_Lat", "Latitud inicial del incidente", "Numérica continua", "Grados, entre -90 y 90", "Ubicación geográfica"),
    ("Start_Lng", "Longitud inicial del incidente", "Numérica continua", "Grados, entre -180 y 180", "Ubicación geográfica"),
    ("Distance(mi)", "Longitud de vía afectada", "Numérica continua", "Millas, mayor o igual a 0", "Aproxima extensión del impacto"),
    ("City", "Ciudad reportada", "Categórica nominal", "Nombre de ciudad", "Segmentación territorial"),
    ("County", "Condado reportado", "Categórica nominal", "Nombre de condado", "Segmentación territorial"),
    ("State", "Código postal del estado", "Categórica nominal", "Código estatal de dos letras", "Comparación geográfica"),
    ("Timezone", "Zona horaria asociada", "Categórica nominal", "Zonas de EE. UU.", "Contextualiza la hora local"),
    ("Weather_Timestamp", "Momento de observación meteorológica", "Temporal", "Fecha-hora convertible", "Evalúa cercanía temporal del clima"),
    ("Temperature(F)", "Temperatura observada", "Numérica continua", "Grados Fahrenheit", "Describe el entorno climático"),
    ("Humidity(%)", "Humedad relativa", "Numérica continua", "Porcentaje entre 0 y 100", "Describe el entorno climático"),
    ("Visibility(mi)", "Visibilidad observada", "Numérica continua", "Millas, mayor o igual a 0", "Condición potencial de conducción"),
    ("Wind_Speed(mph)", "Velocidad del viento", "Numérica continua", "Millas por hora, no negativa", "Describe el entorno climático"),
    ("Precipitation(in)", "Precipitación observada", "Numérica continua", "Pulgadas, no negativa", "Describe el entorno climático"),
    ("Weather_Condition", "Descripción de la condición climática", "Categórica nominal", "Categorías reportadas", "Compara contextos meteorológicos"),
    ("Junction", "Intersección cercana", "Booleana", "True, False o nulo", "Proxy del contexto vial"),
    ("Crossing", "Cruce cercano", "Booleana", "True, False o nulo", "Proxy del contexto vial"),
    ("Traffic_Signal", "Semáforo cercano", "Booleana", "True, False o nulo", "Proxy del contexto vial"),
    ("Stop", "Señal de pare cercana", "Booleana", "True, False o nulo", "Proxy del contexto vial"),
    ("Sunrise_Sunset", "Condición de luz diurna/nocturna", "Categórica nominal", "Day o Night", "Contextualiza la franja horaria"),
    ("impacto_alto", "Indicador derivado: Severity en {3, 4}", "Booleana derivada", "True o False", "Variable central de comparación"),
    ("duracion_horas", "Diferencia entre fin e inicio", "Numérica continua derivada", "Horas, esperada >= 0", "Aproxima duración del incidente"),
    ("hora", "Hora local extraída de Start_Time", "Ordinal cíclica derivada", "0 a 23", "Describe franjas horarias"),
    ("dia_semana", "Día extraído de Start_Time", "Ordinal cíclica derivada", "Lunes a domingo", "Describe patrones semanales"),
], columns=["columna", "descripcion", "tipo_o_escala", "unidad_o_dominio", "importancia"])

display(diccionario_datos)


---
## 7. Preparación de variables analíticas

Las fechas se convierten con `errors="coerce"`: un valor no interpretable pasa a `NaT` y queda visible como problema de validez. Las horas se conservan como hora local reportada por la fuente, útil para decisiones operativas dentro de cada zona.


In [ ]:
# Celda 7.1: fechas y variables derivadas sin eliminar registros
start_raw = df["Start_Time"].astype("string").copy()
end_raw = df["End_Time"].astype("string").copy()
weather_time_raw = df["Weather_Timestamp"].astype("string").copy()

start_reported = start_raw.notna() & start_raw.str.strip().ne("")
end_reported = end_raw.notna() & end_raw.str.strip().ne("")
weather_time_reported = weather_time_raw.notna() & weather_time_raw.str.strip().ne("")

for column in ["Start_Time", "End_Time", "Weather_Timestamp"]:
    df[column] = pd.to_datetime(df[column], errors="coerce", format="mixed")

DAY_ORDER = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
DAY_MAP = dict(enumerate(DAY_ORDER))

df["impacto_alto"] = df["Severity"].isin([3, 4]).astype("boolean")
df["duracion_horas"] = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 3600
df["hora"] = df["Start_Time"].dt.hour.astype("Int8")
df["dia_semana"] = pd.Categorical(
    df["Start_Time"].dt.dayofweek.map(DAY_MAP),
    categories=DAY_ORDER,
    ordered=True,
)
df["mes"] = df["Start_Time"].dt.month.astype("Int8")
df["anio"] = df["Start_Time"].dt.year.astype("Int16")

valid_dates = df["Start_Time"].dropna()
print(f"Fechas válidas de inicio: {len(valid_dates):,} de {len(df):,}")
if not valid_dates.empty:
    print(f"Periodo observado: {valid_dates.min()} → {valid_dates.max()}")
print(f"Accidentes de alto impacto (3–4): {int(df['impacto_alto'].sum()):,}")


---
## 8. EDA orientado a la pregunta

Las comparaciones usan dos medidas distintas:

- **volumen:** número de accidentes registrados;
- **proporción de alto impacto:** porcentaje de severidad 3–4 dentro de los accidentes registrados en cada grupo.

Ninguna de las dos representa por sí sola el riesgo vial de la población, porque el dataset no contiene exposición al tráfico.

### 8.1 Distribución de severidad


In [ ]:
# Celda 8.1: distribución de severidad
severity_counts = df["Severity"].value_counts().reindex([1, 2, 3, 4], fill_value=0)
severity_pct = severity_counts.div(severity_counts.sum()).mul(100)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    severity_counts.index.astype(str),
    severity_counts.values,
    color=[COLORS["green"], COLORS["blue"], COLORS["gold"], COLORS["coral"]],
)
ax.bar_label(bars, labels=[f"{value / 1e6:.2f} M" if value >= 1e6 else f"{value:,}" for value in severity_counts])
ax.set(title="Accidentes registrados por nivel de severidad", xlabel="Severidad", ylabel="Número de accidentes")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
save_fig("01_distribucion_severidad.png", fig)
plt.show()

display(pd.DataFrame({"accidentes": severity_counts, "porcentaje": severity_pct.round(2)}))


### 8.2 ¿Cuándo se concentran los accidentes de alto impacto?

Se comparan horas y días de la semana. La línea/barra de porcentaje responde “de los accidentes registrados en este grupo, ¿qué parte tuvo severidad 3–4?”; no “¿qué probabilidad tiene un conductor de accidentarse?”.


In [ ]:
# Celda 8.2: perfil temporal por hora y día
por_hora = (
    df.dropna(subset=["hora"])
    .groupby("hora", observed=True)
    .agg(total=("impacto_alto", "size"), alto_impacto=("impacto_alto", "sum"))
    .reindex(range(24), fill_value=0)
)
por_hora["alto_impacto_pct"] = np.where(
    por_hora["total"].gt(0), por_hora["alto_impacto"] / por_hora["total"] * 100, np.nan
)

por_dia = (
    df.dropna(subset=["dia_semana"])
    .groupby("dia_semana", observed=True)
    .agg(total=("impacto_alto", "size"), alto_impacto=("impacto_alto", "sum"))
    .reindex(DAY_ORDER, fill_value=0)
)
por_dia["alto_impacto_pct"] = np.where(
    por_dia["total"].gt(0), por_dia["alto_impacto"] / por_dia["total"] * 100, np.nan
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot(por_hora.index, por_hora["alto_impacto_pct"], marker="o", color=COLORS["coral"], linewidth=2)
axes[0].set(title="Proporción de severidad 3–4 por hora", xlabel="Hora local de inicio", ylabel="Alto impacto (%)", xticks=range(0, 24, 2))
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())

axes[1].bar(por_dia.index, por_dia["alto_impacto_pct"], color=COLORS["blue"])
axes[1].set(title="Proporción de severidad 3–4 por día", xlabel="Día de la semana", ylabel="Alto impacto (%)")
axes[1].tick_params(axis="x", rotation=35)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
save_fig("02_patron_temporal_alto_impacto.png", fig)
plt.show()

if por_hora["alto_impacto_pct"].notna().any():
    peak_hour = int(por_hora["alto_impacto_pct"].idxmax())
    print(f"Mayor proporción horaria observada: {peak_hour:02d}:00 ({por_hora.loc[peak_hour, 'alto_impacto_pct']:.2f}%).")
if por_dia["alto_impacto_pct"].notna().any():
    peak_day = por_dia["alto_impacto_pct"].idxmax()
    print(f"Mayor proporción diaria observada: {peak_day} ({por_dia.loc[peak_day, 'alto_impacto_pct']:.2f}%).")


### 8.3 ¿Dónde se concentran?

El panel izquierdo prioriza **carga operativa** (cantidad de accidentes 3–4). El derecho compara la **proporción** de alto impacto y exige un volumen mínimo para no destacar estados con muy pocos registros.


In [ ]:
# Celda 8.3: concentración y proporción por estado
por_estado = (
    df.dropna(subset=["State"])
    .groupby("State", observed=True)
    .agg(total=("impacto_alto", "size"), alto_impacto=("impacto_alto", "sum"))
)
por_estado["alto_impacto_pct"] = por_estado["alto_impacto"] / por_estado["total"] * 100

min_state_obs = max(5, int(len(df) * 0.0005))
estados_elegibles = por_estado.loc[por_estado["total"] >= min_state_obs]
if estados_elegibles.empty:
    estados_elegibles = por_estado

top_volume = por_estado.nlargest(12, "alto_impacto").sort_values("alto_impacto")
top_rate = estados_elegibles.nlargest(12, "alto_impacto_pct").sort_values("alto_impacto_pct")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].barh(top_volume.index.astype(str), top_volume["alto_impacto"], color=COLORS["navy"])
axes[0].set(title="Estados con más accidentes de severidad 3–4", xlabel="Accidentes de alto impacto", ylabel="Estado")
axes[0].xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

axes[1].barh(top_rate.index.astype(str), top_rate["alto_impacto_pct"], color=COLORS["gold"])
axes[1].set(title=f"Mayor proporción de alto impacto\n(mínimo {min_state_obs:,} registros)", xlabel="Alto impacto (%)", ylabel="Estado")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
save_fig("03_patron_geografico_estados.png", fig)
plt.show()

display(por_estado.sort_values("alto_impacto", ascending=False).head(15).round(2))


### 8.4 ¿Bajo qué condiciones climáticas?

Se muestran condiciones con suficiente frecuencia. Una proporción alta puede reflejar clima, localización, hora, fuente de reporte u otros factores; esta comparación no aísla sus efectos.


In [ ]:
# Celda 8.4: volumen y proporción por condición meteorológica
clima = (
    df.assign(condicion_clima=df["Weather_Condition"].astype("string").fillna("No informado"))
    .groupby("condicion_clima", observed=True)
    .agg(total=("impacto_alto", "size"), alto_impacto=("impacto_alto", "sum"))
)
clima["alto_impacto_pct"] = clima["alto_impacto"] / clima["total"] * 100

min_weather_obs = max(5, int(len(df) * 0.0002))
clima_elegible = clima.loc[clima["total"] >= min_weather_obs]
if clima_elegible.empty:
    clima_elegible = clima
top_clima = clima_elegible.nlargest(12, "alto_impacto_pct").sort_values("alto_impacto_pct")

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top_clima.index, top_clima["alto_impacto_pct"], color=COLORS["teal"])
ax.set(
    title=f"Condiciones con mayor proporción de severidad 3–4\n(mínimo {min_weather_obs:,} registros)",
    xlabel="Alto impacto (%)",
    ylabel="Condición meteorológica",
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
for bar, total in zip(bars, top_clima["total"]):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f"  n={total:,}", va="center", fontsize=9)
plt.tight_layout()
save_fig("04_condiciones_climaticas.png", fig)
plt.show()

display(top_clima.sort_values("alto_impacto_pct", ascending=False).round(2))


### 8.5 Contexto vial disponible

Estas banderas indican elementos cercanos al incidente, no un tipo de vía oficial. La comparación “Sí/No” ayuda a detectar asociaciones descriptivas que luego deberían validarse con clase de vía, flujo vehicular y diseño de la intersección.


In [ ]:
# Celda 8.5: proporción de alto impacto según elementos viales cercanos
road_features = [
    "Junction", "Crossing", "Traffic_Signal", "Stop",
    "Railway", "Roundabout", "Traffic_Calming", "Bump",
]
road_rows = []

for feature in road_features:
    valid = df[feature].notna()
    for present, label in [(True, "Sí"), (False, "No")]:
        mask = valid & df[feature].eq(present)
        total = int(mask.sum())
        high = int(df.loc[mask, "impacto_alto"].sum())
        road_rows.append({
            "elemento": feature,
            "presente": label,
            "total": total,
            "alto_impacto": high,
            "alto_impacto_pct": high / total * 100 if total else np.nan,
        })

contexto_vial = pd.DataFrame(road_rows)
fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(
    data=contexto_vial,
    y="elemento",
    x="alto_impacto_pct",
    hue="presente",
    palette={"Sí": COLORS["coral"], "No": COLORS["gray"]},
    ax=ax,
)
ax.set(title="Severidad 3–4 según elemento vial cercano", xlabel="Alto impacto (%)", ylabel="Elemento vial")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title="Elemento presente")
plt.tight_layout()
save_fig("05_contexto_vial.png", fig)
plt.show()

display(contexto_vial.round(2))


---
## 9. Perfil técnico de completitud y duplicados

Antes de interpretar diferencias entre grupos, se cuantifica qué campos faltan y si existen filas o identificadores duplicados. No se elimina ningún registro: esta entrega termina en el diagnóstico.


In [ ]:
# Celda 9.1: inventario técnico por columna
perfil_columnas = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": df.isna().mean().mul(100).round(2),
    "valores_unicos": df.nunique(dropna=True),
}).sort_values(["nulos_pct", "nulos"], ascending=False)

display(perfil_columnas)

exact_duplicates = int(df.duplicated().sum())
id_text = df["ID"].astype("string").str.strip()
id_present = id_text.notna() & id_text.ne("")
duplicated_ids = int(id_text[id_present].duplicated(keep=False).sum())

print(f"Filas exactamente duplicadas: {exact_duplicates:,} ({exact_duplicates / len(df) * 100:.4f}%)")
print(f"Filas involucradas en ID duplicado: {duplicated_ids:,} ({duplicated_ids / len(df) * 100:.4f}%)")


---
## 10. Evaluación de calidad con reglas justificadas

Se evalúan las cuatro dimensiones solicitadas por la rúbrica:

- **Completitud:** presencia de campos indispensables para la pregunta.
- **Unicidad:** cada `ID` informado debe aparecer una sola vez.
- **Validez:** dominios, rangos físicos y fechas convertibles.
- **Consistencia:** relaciones lógicas entre campos, especialmente inicio y fin.

Cada regla conserva su denominador. Por ejemplo, la consistencia temporal se calcula solo cuando ambas fechas pudieron interpretarse; los faltantes se reportan por separado como completitud/validez.


In [ ]:
# Celda 10.1: función reusable y reglas de calidad
quality_results = []


def registrar_regla(
    dimension,
    regla,
    campo,
    cumple,
    impacto,
    accion,
    responsable,
    aplica=None,
):
    cumple = pd.Series(cumple, index=df.index, dtype="boolean")
    if aplica is None:
        aplica = pd.Series(True, index=df.index, dtype="boolean")
    else:
        aplica = pd.Series(aplica, index=df.index, dtype="boolean").fillna(False)

    evaluados = int(aplica.sum())
    incumple = aplica & ~cumple.fillna(False)
    incumplimientos = int(incumple.sum())
    cumplimiento_pct = (
        (evaluados - incumplimientos) / evaluados * 100 if evaluados else np.nan
    )
    quality_results.append({
        "dimension": dimension,
        "regla": regla,
        "campo": campo,
        "evaluados": evaluados,
        "incumplimientos": incumplimientos,
        "cumplimiento_pct": round(cumplimiento_pct, 2),
        "afectados_pct_total": round(incumplimientos / len(df) * 100, 4),
        "impacto": impacto,
        "accion_propuesta": accion,
        "responsable_sugerido": responsable,
    })


critical_complete = (
    id_present
    & df["Severity"].notna()
    & start_reported
    & df["State"].notna()
)
weather_complete = (
    df["Weather_Condition"].notna()
    & df["Visibility(mi)"].notna()
    & df["Temperature(F)"].notna()
)

registrar_regla(
    "Completitud", "Campos críticos informados", "ID / Severity / Start_Time / State",
    critical_complete,
    "Impide ubicar el reporte en tiempo, severidad o territorio.",
    "Solicitar campos obligatorios y rechazar registros incompletos en la ingesta.",
    "Data Owner / Data Steward",
)
registrar_regla(
    "Completitud", "Contexto meteorológico mínimo informado",
    "Weather_Condition / Visibility(mi) / Temperature(F)", weather_complete,
    "Reduce la muestra disponible y puede sesgar comparaciones por clima.",
    "Monitorear cobertura por fuente, estado y año; no imputar sin estudiar el patrón de ausencia.",
    "Data Steward / Proveedor de datos",
)
registrar_regla(
    "Unicidad", "ID informado aparece una sola vez", "ID",
    ~id_text.duplicated(keep=False),
    "Los duplicados inflan conteos y cambian porcentajes de alto impacto.",
    "Investigar duplicados por fuente y conservar una regla documentada de deduplicación.",
    "Data Steward / Custodian TI", aplica=id_present,
)
registrar_regla(
    "Validez", "Severity pertenece a {1, 2, 3, 4}", "Severity",
    df["Severity"].isin([1, 2, 3, 4]),
    "Clasificaciones fuera del dominio invalidan el indicador de alto impacto.",
    "Validar el dominio en la ingesta y aislar valores no reconocidos.",
    "Data Owner / Data Steward",
)
registrar_regla(
    "Validez", "Start_Time informado es convertible", "Start_Time",
    df["Start_Time"].notna(),
    "Fechas inválidas impiden asignar hora, día y periodo.",
    "Estandarizar formato y zona horaria en el sistema fuente.",
    "Custodian TI / Proveedor de datos", aplica=start_reported,
)
registrar_regla(
    "Validez", "End_Time informado es convertible", "End_Time",
    df["End_Time"].notna(),
    "Fechas inválidas impiden estimar la duración del incidente.",
    "Estandarizar formato y documentar el significado del cierre.",
    "Custodian TI / Proveedor de datos", aplica=end_reported,
)

valid_state_codes = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC",
}
registrar_regla(
    "Validez", "State es un código estatal reconocido", "State",
    df["State"].astype("string").isin(valid_state_codes),
    "Códigos no reconocidos generan ubicaciones erróneas o grupos artificiales.",
    "Contrastar contra el catálogo oficial y corregir en el origen.",
    "Data Steward", aplica=df["State"].notna(),
)
registrar_regla(
    "Validez", "Coordenadas dentro de rangos físicos", "Start_Lat / Start_Lng",
    df["Start_Lat"].between(-90, 90) & df["Start_Lng"].between(-180, 180),
    "Coordenadas imposibles distorsionan cualquier análisis geográfico.",
    "Validar rangos y contrastar estado/condado antes de publicar mapas.",
    "Data Steward / Custodian TI",
)
registrar_regla(
    "Validez", "Distance(mi) no es negativa", "Distance(mi)",
    df["Distance(mi)"].ge(0),
    "Distancias negativas vuelven incoherente la extensión del impacto.",
    "Aplicar restricción no negativa y revisar la unidad de medición.",
    "Data Steward / Custodian TI", aplica=df["Distance(mi)"].notna(),
)
registrar_regla(
    "Validez", "Humidity(%) está entre 0 y 100", "Humidity(%)",
    df["Humidity(%)"].between(0, 100),
    "Valores imposibles degradan las comparaciones meteorológicas.",
    "Validar rango físico y revisar sensores/unidades.",
    "Proveedor meteorológico / Data Steward", aplica=df["Humidity(%)"].notna(),
)
registrar_regla(
    "Validez", "Visibility(mi) no es negativa", "Visibility(mi)",
    df["Visibility(mi)"].ge(0),
    "Visibilidad negativa carece de interpretación física.",
    "Aplicar restricción no negativa y revisar valores extremos.",
    "Proveedor meteorológico / Data Steward", aplica=df["Visibility(mi)"].notna(),
)
registrar_regla(
    "Consistencia", "End_Time no precede a Start_Time", "Start_Time / End_Time",
    df["End_Time"].ge(df["Start_Time"]),
    "Duraciones negativas alteran la priorización de incidentes prolongados.",
    "Revisar zonas horarias y reglas de cierre; corregir en la fuente, no por descarte automático.",
    "Data Steward / Proveedor de datos",
    aplica=df["Start_Time"].notna() & df["End_Time"].notna(),
)
registrar_regla(
    "Consistencia", "Weather_Timestamp no difiere más de 24 h del inicio",
    "Weather_Timestamp / Start_Time",
    (df["Weather_Timestamp"] - df["Start_Time"]).abs().dt.total_seconds().le(24 * 3600),
    "Un clima lejano en el tiempo puede no representar las condiciones del accidente.",
    "Documentar tolerancia temporal y priorizar observaciones meteorológicas cercanas.",
    "Data Owner / Proveedor meteorológico",
    aplica=df["Weather_Timestamp"].notna() & df["Start_Time"].notna(),
)

resultado_calidad = pd.DataFrame(quality_results).sort_values(
    ["cumplimiento_pct", "incumplimientos"], ascending=[True, False], na_position="last"
).reset_index(drop=True)

display(resultado_calidad)
resultado_calidad.to_csv(
    OUTPUT_DIR / "diagnostico_calidad.csv", index=False, encoding="utf-8-sig"
)


In [ ]:
# Celda 10.2: resumen visual de cumplimiento
quality_plot = resultado_calidad.dropna(subset=["cumplimiento_pct"]).sort_values("cumplimiento_pct")

fig, ax = plt.subplots(figsize=(11, 8))
dimension_colors = {
    "Completitud": COLORS["blue"],
    "Unicidad": COLORS["green"],
    "Validez": COLORS["gold"],
    "Consistencia": COLORS["coral"],
}
bars = ax.barh(
    quality_plot["regla"],
    quality_plot["cumplimiento_pct"],
    color=quality_plot["dimension"].map(dimension_colors),
)
ax.axvline(95, color=COLORS["gray"], linestyle="--", linewidth=1, label="Referencia descriptiva: 95%")
ax.set(title="Cumplimiento de reglas de calidad", xlabel="Cumplimiento (%)", ylabel="Regla", xlim=(0, 101))
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(loc="lower right")
for bar, value in zip(bars, quality_plot["cumplimiento_pct"]):
    ax.text(min(value + 0.6, 99.5), bar.get_y() + bar.get_height() / 2, f"{value:.2f}%", va="center", fontsize=8)
plt.tight_layout()
save_fig("06_calidad_datos.png", fig)
plt.show()


---
## 11. Diagnóstico final priorizado

La priorización selecciona hasta tres reglas con menor cumplimiento y al menos un incumplimiento. Para cada una responde:

1. ¿Qué problema se encontró?
2. ¿Cuántos registros y qué porcentaje afecta?
3. ¿Cómo puede alterar la descripción o la decisión?
4. ¿Qué acción se propone y quién debería atenderla?


In [ ]:
# Celda 11.1: ficha final con problemas, impacto, acción y responsable
hallazgos_con_error = resultado_calidad.loc[
    resultado_calidad["incumplimientos"].gt(0)
    & resultado_calidad["cumplimiento_pct"].notna()
]
diagnostico_final = hallazgos_con_error.head(3).copy()

if diagnostico_final.empty:
    print("No se detectaron incumplimientos en las reglas evaluadas.")
    print("Esto no demuestra ausencia de otros problemas: amplía reglas y contrasta con la fuente.")
else:
    diagnostico_final["problema_cuantificado"] = diagnostico_final.apply(
        lambda row: (
            f"{row['incumplimientos']:,} registros "
            f"({row['afectados_pct_total']:.4f}% del total cargado)"
        ),
        axis=1,
    )
    display(diagnostico_final[[
        "dimension", "regla", "problema_cuantificado", "cumplimiento_pct",
        "impacto", "accion_propuesta", "responsable_sugerido",
    ]])

    print("DIAGNÓSTICO EJECUTIVO")
    print("=" * 80)
    for number, (_, row) in enumerate(diagnostico_final.iterrows(), start=1):
        print(f"\n{number}. Problema: {row['regla']} ({row['dimension']}).")
        print(
            f"   Magnitud: {row['incumplimientos']:,} registros; "
            f"{row['afectados_pct_total']:.4f}% del total; "
            f"cumplimiento de {row['cumplimiento_pct']:.2f}%."
        )
        print(f"   Impacto: {row['impacto']}")
        print(f"   Acción: {row['accion_propuesta']}")
        print(f"   Responsable: {row['responsable_sugerido']}")

    diagnostico_final.to_csv(
        OUTPUT_DIR / "diagnostico_priorizado.csv", index=False, encoding="utf-8-sig"
    )


---
## 12. Limitaciones y uso responsable

1. **Proxy, no colapso observado:** severidad 3–4 representa impacto reportado en el tráfico; el dataset no mide longitud de cola, retraso total ni tiempo de respuesta de ambulancias.
2. **Sin denominador de exposición:** los conteos y proporciones son entre accidentes registrados; no estiman el riesgo por conductor, viaje o kilómetro.
3. **Sesgo de cobertura y reporte:** fuentes, estados y años pueden tener distinta capacidad para detectar o transmitir incidentes.
4. **Clima aproximado:** las mediciones meteorológicas pueden provenir de otra hora o estación; por eso se evalúa la cercanía temporal.
5. **Contexto vial incompleto:** las banderas de elementos cercanos no sustituyen una clasificación de vía ni el volumen vehicular.
6. **Asociación no es causalidad:** una diferencia por hora, clima o estado puede depender de factores no observados.

### Uso sugerido para la decisión

Los resultados sirven para **formular franjas y zonas candidatas** a mayor vigilancia o capacidad de respuesta. Antes de reasignar recursos deben contrastarse con volumen de tráfico, obras viales, población, estacionalidad y validación de expertos locales.


In [ ]:
# Celda 12.1: registro mínimo de trazabilidad de esta ejecución
registro_trazabilidad = pd.Series({
    "dataset": "sobhanmoosavi/us-accidents",
    "archivo_fuente": CSV_PATH.name,
    "ruta_relativa_o_externa": (
        str(CSV_PATH.relative_to(PROJECT_ROOT))
        if CSV_PATH.is_relative_to(PROJECT_ROOT)
        else str(CSV_PATH)
    ),
    "tamano_bytes": CSV_PATH.stat().st_size,
    "filas_leidas": len(df),
    "columnas_originales_seleccionadas": len(USECOLS),
    "alcance_lectura": load_scope,
    "ejecucion_utc": datetime.now(timezone.utc).isoformat(),
    "transformaciones": "tipado; conversión de fechas; variables derivadas; EDA; reglas de calidad",
    "filas_eliminadas": 0,
}, name="valor").to_frame()

display(registro_trazabilidad)


---
## 13. Checklist de entrega

- [x] Tema, propósito y pregunta motivadora.
- [x] Fuente, periodo, tamaño, archivo y condiciones de uso.
- [x] Unidad de análisis y significado de una fila.
- [x] Carga reproducible sin rutas dependientes de una computadora específica.
- [x] Evidencia con `head()`, `shape`, `info()` y `describe()` e interpretación.
- [x] Diccionario de columnas relevantes.
- [x] Reglas justificadas de completitud, unicidad, validez y consistencia.
- [x] Problemas cuantificados, impacto, acción y responsable.
- [x] Limitaciones y advertencia contra conclusiones causales.
- [ ] Completar los nombres de los integrantes.

> Primero comprendemos el problema y los datos. Después evaluamos si la evidencia tiene la calidad necesaria para sostener el análisis.
